# Member A — Dunnhumby Data Transformation

Transforms the 8 **Dunnhumby: The Complete Journey** CSVs (Segments 6/7/8: Replenishment-Ready, Cross-Sell Opportunity, Seasonal Purchase) into canonical-schema CSVs (`backend/schema.sql`).

**Spec:** `DUNNHUMBY.md` — mappings with citations, gaps, and open questions.
**Logic:** `common.py` (id namespace + per-dataset date strategy), `dunnhumby.py` (mappings/cleaning), `validate.py` (71 constraint checks).

This notebook orchestrates those modules — the transformation itself is not duplicated here, so there is one source of truth.

Raw files: `../raw/dunnhumby/` · Output: `../processed/dunnhumby/`

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from common import uid
from dunnhumby import ANCHOR, COLUMNS, RAW_FILES, load_raw, transform
from validate import validate

RAW_DIR = Path('../raw/dunnhumby')
OUT_DIR = Path('../processed/dunnhumby')
LOAD_ORDER = ['stores', 'customers', 'products', 'orders', 'order_items',
              'coupon_campaigns', 'coupon_redemptions', 'causal_data']
STORE_ID = uid('stores', 'dunnhumby-demo')   # deterministic tenant placeholder

print(f'anchor  : {ANCHOR}')
print(f'source  : {RAW_DIR}')
print(f'output  : {OUT_DIR}')
print(f'sources : {", ".join(RAW_FILES)}')

In [ ]:
raw = load_raw(RAW_DIR)
for name, df in raw.items():
    print(f'{name:<24} {len(df):>12,} rows')

In [ ]:
started = time.time()
result = transform(raw, STORE_ID)
print(f'transformed in {time.time() - started:.1f}s\n')
for table in LOAD_ORDER:
    print(f'  {table:<20} {len(result.frames[table]):>10,} rows')

In [ ]:
report = validate(result.frames, result.raw_counts, result.dropped,
                  result.dropped_by_source, anchor=ANCHOR)
print(report.render())

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for table in LOAD_ORDER:
    result.frames[table][COLUMNS[table]].to_csv(
        OUT_DIR / f'{table}.csv', index=False, na_rep=''
    )
print(f'wrote {len(LOAD_ORDER)} canonical CSVs to {OUT_DIR}')